# Generación stops desde shapelines 

In [58]:
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import Point
from sklearn.cluster import DBSCAN

In [59]:
ciudad = "tampico"

In [60]:
path_base = f"./data/proc/{ciudad}/"


___

In [61]:
path_gtfs = "../1-scraping_ruta_directa/data/proc/rutas_procesadas_tampico_1.geojson"

In [63]:
routes = gpd.read_file(path_gtfs)
print(routes["data.route.shortName"].nunique())
routes.head()

121


,data.route.shortName,data.route.longName,shape_id,geometry
0,74,Isleta Pérez,shape_74,"LINESTRING (-97.84481 22.20913, -97.84525 22.2..."
1,76,Cascajal,shape_76,"LINESTRING (-97.87153 22.22684, -97.87129 22.2..."
2,81,Golfo,shape_81,"LINESTRING (-97.83941 22.22638, -97.84048 22.2..."
3,89A,Altamira - Tampiquito por Soriana,shape_89A,"LINESTRING (-97.92128 22.41213, -97.92228 22.4..."
4,67,Central Camionera - Madero,shape_67,"LINESTRING (-97.85804 22.25045, -97.85752 22.2..."


In [64]:
# routes = routes[routes["data.route.shortName"]=="119"]

In [65]:
routes.explore(column="shape_id", categorical=True)

## Generación de stops

🔹 Opción 1: Muestreo a intervalos fijos sobre la geometría

	1.	Usar la longitud de cada LINESTRING.
	2.	Interpolar un punto cada N metros.
	3.	Guardar esos puntos como stops.

In [66]:
import math
import numpy as np
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point
from shapely.ops import linemerge

def _as_linestring(geom):
    if isinstance(geom, LineString):
        return geom
    if isinstance(geom, MultiLineString):
        merged = linemerge(geom)
        if isinstance(merged, MultiLineString):
            # si no se puede unir en una sola, toma la más larga
            return max(merged.geoms, key=lambda g: g.length)
        return merged
    raise ValueError(f"Se esperaba LineString/MultiLineString, llegó {geom.geom_type}")

def generar_stops_laterales_por_ruta(
    gdf, distancia=100, offset=3, include_center=False
):
    """
    Por cada ruta (LineString/MultiLineString) genera puntos cada 'distancia' metros
    y duplica cada punto a los lados 'left' y 'right' (perpendicular a la línea) a 'offset' metros.
    Reproyecta a CRS métrico si es necesario y devuelve en el CRS original.

    Requiere columnas: ['data.route.shortName', 'geometry'].
    """
    if gdf.crs is None:
        raise ValueError("El GeoDataFrame no tiene CRS. Asigna uno (p.ej. EPSG:4326) antes.")

    orig_crs = gdf.crs
    # A métrico si hace falta
    if not gdf.crs.is_projected:
        try:
            gdfm = gdf.to_crs(gdf.estimate_utm_crs())
        except Exception:
            gdfm = gdf.to_crs("EPSG:3857")  # fallback
    else:
        gdfm = gdf.copy()

    rows = []
    for _, row in gdfm.iterrows():
        route_id = row["data.route.shortName"]
        line = _as_linestring(row.geometry)
        if line.is_empty:
            continue

        L = float(line.length)
        if L == 0:
            # un punto degenerado
            p = line.interpolate(0)
            if include_center:
                rows.append({"route_id": route_id, "stop_id": f"{route_id}_center0", "side": "center", "geometry": p})
            # normales arbitrarias si no hay dirección
            nx, ny = 0.0, 1.0
            left_p  = Point(p.x + offset*nx, p.y + offset*ny)
            right_p = Point(p.x - offset*nx, p.y - offset*ny)
            rows.append({"route_id": route_id, "stop_id": f"{route_id}_left0",  "side": "left",  "geometry": left_p})
            rows.append({"route_id": route_id, "stop_id": f"{route_id}_right0", "side": "right", "geometry": right_p})
            continue

        # distancias: 0, N, 2N, ... < L y añadimos L
        dists = list(np.arange(0.0, L, float(distancia)))
        if not math.isclose(dists[-1] if dists else -1.0, L, rel_tol=1e-9, abs_tol=1e-6):
            dists.append(L)

        for d in dists:
            p = line.interpolate(d)

            # Paso para derivada/tangente (pequeño, en metros)
            # usa algo proporcional a offset y a la longitud
            h = max(min(offset*0.5, 2.0), min(1.0, L*0.01))
            a = max(d - h, 0.0)
            b = min(d + h, L)
            if math.isclose(a, b, abs_tol=1e-9):
                # en extremos muy cortos, usa un paso mínimo
                b = min(d + max(h, 0.5), L)
                a = max(d - max(h, 0.5), 0.0)

            pa = line.interpolate(a)
            pb = line.interpolate(b)

            ux, uy = (pb.x - pa.x), (pb.y - pa.y)
            norm = math.hypot(ux, uy)
            if norm == 0.0:
                # si no hay dirección local (segmento plano), elige normal vertical
                tx, ty = 1.0, 0.0
            else:
                tx, ty = ux/norm, uy/norm

            # normal a la izquierda (+90°): n = (-ty, tx)
            nx, ny = -ty, tx

            left_point  = Point(p.x + offset*nx, p.y + offset*ny)
            right_point = Point(p.x - offset*nx, p.y - offset*ny)

            # nombres cómodos para inicio/fin o índice
            if math.isclose(d, 0.0, abs_tol=1e-6):
                tag = "start"
            elif math.isclose(d, L,  abs_tol=1e-6):
                tag = "end"
            else:
                tag = str(int(round(d/float(distancia))))

            if include_center:
                rows.append({"route_id": route_id, "stop_id": f"{route_id}_center_{tag}", "side": "center", "geometry": p})

            rows.append({"route_id": route_id, "stop_id": f"{route_id}_left_{tag}",  "side": "left",  "geometry": left_point})
            rows.append({"route_id": route_id, "stop_id": f"{route_id}_right_{tag}", "side": "right", "geometry": right_point})

    out = gpd.GeoDataFrame(rows, crs=gdfm.crs)
    if out.crs != orig_crs:
        out = out.to_crs(orig_crs)
    return out

## Limpieza de stops

In [67]:
import pandas as pd
import geopandas as gpd
import numpy as np
from sklearn.cluster import DBSCAN
import math

# ======================================================
# 1) Deduplicar paradas por "side" con DBSCAN
#    - Ejecuta DBSCAN por cada categoría de `side`
#    - El representante de cada clúster es el punto más cercano
#      al centroide del clúster (determinístico)
#    - Importante: `eps` está en unidades del CRS (usa metros)
# ======================================================
def deduplicar_stops(gdf, eps=30, side_col="side"):
    if gdf.empty:
        return gdf.copy()

    if side_col not in gdf.columns:
        # Sin columna 'side', aplica una sola pasada
        return _dedup_single_group(gdf, eps)

    partes = []
    for side_value, sub in gdf.groupby(side_col):
        dedup_side = _dedup_single_group(sub, eps)
        partes.append(dedup_side)

    out = gpd.GeoDataFrame(pd.concat(partes, ignore_index=True), crs=gdf.crs)
    return out


def _dedup_single_group(gdf, eps):
    # Ejecuta DBSCAN en un subconjunto (una categoría de `side` o el total)
    coords = np.c_[gdf.geometry.x.values, gdf.geometry.y.values]
    labels = DBSCAN(eps=eps, min_samples=1).fit(coords).labels_

    g = gdf.copy()
    g.loc[:, "cluster"] = labels

    reps = []
    for cl, grp in g.groupby("cluster"):
        # Centroide del clúster (en 2D)
        cx = grp.geometry.x.mean()
        cy = grp.geometry.y.mean()
        # índice del punto más cercano al centroide (determinístico)
        d2 = (grp.geometry.x - cx)**2 + (grp.geometry.y - cy)**2
        idx_rep = d2.idxmin()
        reps.append(g.loc[idx_rep])

    reps_gdf = gpd.GeoDataFrame(reps, crs=gdf.crs).reset_index(drop=True)
    return reps_gdf


# ======================================================
# 2) Deduplicar en batches (pares de rutas por defecto)
#    - Dentro de cada batch, deduplica por `side`
# ======================================================
def batch_deduplicar_pairs(gdf_stops, batch_size=2, eps=30, side_col="side"):
    """
    Procesa las rutas en lotes de `batch_size` (2 por defecto).
    Para cada lote aplica deduplicación por 'side' y devuelve
    una lista de GeoDataFrames (uno por batch).
    """
    rutas = gdf_stops["route_id"].unique()
    n_batches = math.ceil(len(rutas) / batch_size)

    results = []
    for i in range(n_batches):
        rutas_batch = rutas[i*batch_size : (i+1)*batch_size]
        gdf_batch = gdf_stops[gdf_stops["route_id"].isin(rutas_batch)].copy()

        dedup = deduplicar_stops(gdf_batch, eps=eps, side_col=side_col)
        dedup["batch"] = i
        results.append(dedup)

    return results


# ======================================================
# 3) Deduplicación iterativa de resultados
#    - En cada paso concatena varios resultados y vuelve a
#      deduplicar por `side`, hasta quedar en uno solo.
# ======================================================
def iterative_dedup(results, group_size=2, eps=30, side_col="side"):
    """
    Aplica deduplicación iterativa sobre una lista de GeoDataFrames
    hasta obtener un único resultado. Deduplica por 'side' en cada paso.
    """
    if not results:
        return gpd.GeoDataFrame(geometry=[], crs=None)

    step = 0
    while len(results) > 1:
        new_results = []
        n_groups = math.ceil(len(results) / group_size)

        for i in range(n_groups):
            block = results[i*group_size : (i+1)*group_size]

            merged_block = gpd.GeoDataFrame(
                pd.concat(block, ignore_index=True),
                crs=results[0].crs
            )

            dedup_block = deduplicar_stops(merged_block, eps=eps, side_col=side_col)
            dedup_block["iter_step"] = step
            dedup_block["block"] = i
            new_results.append(dedup_block)

        results = new_results
        step += 1
        print(f"Iteración {step}: {len(results)} resultados")

    return results[0]

In [68]:
# 0) Generar stops laterales (queda en el CRS original de 'routes')
gdf_stops = generar_stops_laterales_por_ruta(
    routes, distancia=250, offset=3, include_center=False
)

In [69]:
gdf_stops = gdf_stops[gdf_stops["side"]=="left"]

In [70]:

# 1) Reproyectar a CRS MÉTRICO (para que eps=30 sean 30 METROS)
metric_crs = routes.estimate_utm_crs()  # o fija uno: "EPSG:32614" p.ej.
gdf_stops_m = gdf_stops.to_crs(metric_crs)

# 2) Deduplicación por batches y por 'side'
results = batch_deduplicar_pairs(gdf_stops_m, batch_size=2, eps=125, side_col="side")

# 3) Combinar iterativamente (sigue deduplicando por 'side')
final_m = iterative_dedup(results, group_size=2, eps=125, side_col="side")

# 4) Volver al CRS original (opcional)
gdf_stops = final_m.to_crs(routes.crs)

Iteración 1: 31 resultados
Iteración 2: 16 resultados
Iteración 3: 8 resultados
Iteración 4: 4 resultados
Iteración 5: 2 resultados
Iteración 6: 1 resultados


In [71]:
gdf_stops["stop_id_ix"] = gdf_stops["stop_id"].str.split("_").str[-1]
gdf_stops["stop_id_ix"] = gdf_stops["stop_id_ix"].astype(float, errors="ignore")

# Try to convert to float, setting invalid parsing to NaN
gdf_stops["stop_id_ix_float"] = pd.to_numeric(gdf_stops["stop_id_ix"], errors="coerce")

# Compute max of valid floats
max_val = gdf_stops["stop_id_ix_float"].max()

# Replace "end" (or any non-float value) with the max value
gdf_stops["stop_id_ix_float"] = gdf_stops["stop_id_ix_float"].fillna(max_val)

# Drop any remaining NaN (if exist) and cast to float
gdf_stops = gdf_stops.dropna(subset=["stop_id_ix_float"]).astype({"stop_id_ix_float": float})

gdf_stops

,route_id,stop_id,side,geometry,cluster,batch,iter_step,block,stop_id_ix,stop_id_ix_float
0,74,74_left_end,left,POINT (-97.84477 22.2092),0,0,5,0,end,457.0
1,74,74_left_1,left,POINT (-97.84393 22.21058),1,0,5,0,1,1.0
2,74,74_left_2,left,POINT (-97.84203 22.21182),2,0,5,0,2,2.0
3,74,74_left_15,left,POINT (-97.846 22.21088),3,0,5,0,15,15.0
4,68,68_left_48,left,POINT (-97.84904 22.21329),4,11,5,0,48,48.0
...,...,...,...,...,...,...,...,...,...,...
1533,108,108_left_429,left,POINT (-98.11371 22.50806),1533,60,5,0,429,429.0
1534,108,108_left_423,left,POINT (-98.10389 22.49846),1534,60,5,0,423,423.0
1535,108,108_left_58,left,POINT (-98.05562 22.49689),1535,60,5,0,58,58.0
1536,108,108_left_63,left,POINT (-98.04411 22.49326),1536,60,5,0,63,63.0


In [72]:
gdf_stops.head()

,route_id,stop_id,side,geometry,cluster,batch,iter_step,block,stop_id_ix,stop_id_ix_float
0,74,74_left_end,left,POINT (-97.84477 22.2092),0,0,5,0,end,457.0
1,74,74_left_1,left,POINT (-97.84393 22.21058),1,0,5,0,1,1.0
2,74,74_left_2,left,POINT (-97.84203 22.21182),2,0,5,0,2,2.0
3,74,74_left_15,left,POINT (-97.846 22.21088),3,0,5,0,15,15.0
4,68,68_left_48,left,POINT (-97.84904 22.21329),4,11,5,0,48,48.0


In [73]:
# routes = routes[routes["data.route.shortName"]=="119"]
# gdf_stops = gdf_stops[gdf_stops["route_id"]=="119"]

In [74]:
# gdf_stops.to_file("./temp_stops.geojson")
#routes.to_file("./temp_routes.geojson")

In [75]:
m = routes.explore()
gdf_stops[gdf_stops["side"]=="left"].explore(m=m,  column="stop_id_ix_float", categorical=False)

## Generar tabla stops

In [76]:
def format_stops_for_gtfs(gdf):
    """
    Convierte un GeoDataFrame de stops deduplicados
    al formato de stops.txt de GTFS.
    """
    gdf = gdf.to_crs(epsg=4326)  # asegurarse de estar en lat/lon WGS84
    
    # crear columnas GTFS
    df = pd.DataFrame({
        "stop_code": [str(i).zfill(4) for i in range(len(gdf))],
        "stop_lat": gdf.geometry.y,
        "stop_lon": gdf.geometry.x,
        "stop_id": [f"stop_{s}_{str(i+1).zfill(4)}" for i, s in enumerate(gdf["side"])],
        "stop_name": [f"Stop {i+1}" for i in range(len(gdf))],
        "zone_id": ["merged_" + str(c) for c in gdf["cluster"]],
        "parent_station": "",
        "stop_desc": "",
        "location_type": 0
    })
    
    return df

In [77]:
# solo si no se hace iterativo
gdf_stops["cluster"] = "cluster"
formated_table = format_stops_for_gtfs(gdf_stops)
formated_table.head(10)

,stop_code,stop_lat,stop_lon,stop_id,stop_name,zone_id,parent_station,stop_desc,location_type
0,0000,22.209196,-97.844766,stop_left_0001,Stop 1,merged_cluster,,,0
1,0001,22.210581,-97.843926,stop_left_0002,Stop 2,merged_cluster,,,0
2,0002,22.211820,-97.842028,stop_left_0003,Stop 3,merged_cluster,,,0
3,0003,22.210877,-97.846005,stop_left_0004,Stop 4,merged_cluster,,,0
4,0004,22.213285,-97.849042,stop_left_0005,Stop 5,merged_cluster,,,0
5,0005,22.212175,-97.849720,stop_left_0006,Stop 6,merged_cluster,,,0
6,0006,22.214650,-97.851593,stop_left_0007,Stop 7,merged_cluster,,,0
7,0007,22.214623,-97.853656,stop_left_0008,Stop 8,merged_cluster,,,0
8,0008,22.212535,-97.855752,stop_left_0009,Stop 9,merged_cluster,,,0
9,0009,22.213634,-97.858347,stop_left_0010,Stop 10,merged_cluster,,,0


In [78]:
formated_table.shape

(1538, 9)

In [79]:
formated_table = format_stops_for_gtfs(gdf_stops)
formated_table.head(10)

,stop_code,stop_lat,stop_lon,stop_id,stop_name,zone_id,parent_station,stop_desc,location_type
0,0000,22.209196,-97.844766,stop_left_0001,Stop 1,merged_cluster,,,0
1,0001,22.210581,-97.843926,stop_left_0002,Stop 2,merged_cluster,,,0
2,0002,22.211820,-97.842028,stop_left_0003,Stop 3,merged_cluster,,,0
3,0003,22.210877,-97.846005,stop_left_0004,Stop 4,merged_cluster,,,0
4,0004,22.213285,-97.849042,stop_left_0005,Stop 5,merged_cluster,,,0
5,0005,22.212175,-97.849720,stop_left_0006,Stop 6,merged_cluster,,,0
6,0006,22.214650,-97.851593,stop_left_0007,Stop 7,merged_cluster,,,0
7,0007,22.214623,-97.853656,stop_left_0008,Stop 8,merged_cluster,,,0
8,0008,22.212535,-97.855752,stop_left_0009,Stop 9,merged_cluster,,,0
9,0009,22.213634,-97.858347,stop_left_0010,Stop 10,merged_cluster,,,0


In [80]:
formated_table.to_csv(path_base + "stops.csv", index=False)

In [81]:
path_base

'./data/proc/tampico/'